# QCar PyramidFuse — Training Walkthrough

This notebook follows `CP_Fusion_MODELS.ipynb` and completes the PyramidFusion training path.

**Important:** no full camera-Pyramid checkpoint compatible with this HEAL architecture is available. The notebook can transfer the compatible AttFuse LSS encoder, but the Pyramid backbone, occupancy heads, and fusion weights begin randomly initialized. Treat this as an experiment, not a ready pretrained baseline.

## Architecture
images → LSS → small per-agent BEV backbone → aligner → multiscale PyramidFusion (occupancy-weighted affine fusion) → decoder → post-fusion shrinker → heads → detection + pyramid occupancy losses.

## Phase 0 — Environment and explicit experimental opt-in

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import os, random, statistics, json, gc, copy
from datetime import datetime
import numpy as np
import torch
from torch.utils.data import DataLoader

HEAL_ROOT = Path.cwd()
if not (HEAL_ROOT / 'opencood').is_dir(): HEAL_ROOT = (HEAL_ROOT / '..').resolve()
os.chdir(HEAL_ROOT)
import qcar.patches.patch_1cam_loader
import opencood.hypes_yaml.yaml_utils as yaml_utils
from opencood.data_utils.datasets import build_dataset
from opencood.tools import train_utils

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), 'Full HEAL training requires CUDA.'
DEVICE = torch.device('cuda')
ALLOW_EXPERIMENTAL_PYRAMID_TRAINING = False
print('Set ALLOW_EXPERIMENTAL_PYRAMID_TRAINING=True after reading the checkpoint warning.')

## Phase 1 — Load Pyramid configuration and development split

In [ ]:
CONFIG_PATH = 'qcar/configs/camera_pyramid_onlyfront.yaml'
hypes = yaml_utils.load_yaml(CONFIG_PATH, SimpleNamespace(model_dir=''))
hypes['train_params']['batch_size'] = 1
assert hypes['model']['core_method'] == 'heter_pyramid_collab'
assert hypes['_qcar_pretrained_checkpoint'] is None
assert hypes.get('test_dir') is None
print('Train:', hypes['root_dir'])
print('Validate:', hypes['validate_dir'])
print('Full Pyramid checkpoint:', hypes['_qcar_pretrained_checkpoint'])
print('Pyramid loss:', hypes['loss']['args']['pyramid'])

In [ ]:
train_dataset = build_dataset(hypes, visualize=False, train=True)
val_dataset = build_dataset(hypes, visualize=False, train=False)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0,
                          collate_fn=train_dataset.collate_batch_train,
                          pin_memory=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0,
                        collate_fn=val_dataset.collate_batch_train,
                        pin_memory=True, drop_last=False)
presence = [int(val_dataset[i]['ego']['object_bbx_mask'].sum()) > 0
            for i in range(len(val_dataset))]
print('Train:', len(train_dataset), 'Validation:', len(val_dataset),
      'positive:', sum(presence), 'negative:', len(presence)-sum(presence))

## Phase 2 — Instantiate PyramidFusion and transfer only compatible encoder weights
The transfer below initializes `encoder_m2.*` from AttFuse. It does **not** pretend that AttFusion weights fit PyramidFusion.

In [ ]:
assert ALLOW_EXPERIMENTAL_PYRAMID_TRAINING, 'Explicit opt-in required.'
model = train_utils.create_model(hypes)
criterion = train_utils.create_loss(hypes)

ATTFUSE_CHECKPOINT = HEAL_ROOT / 'checkpoints/opv2v_camera/HeterBaseline_opv2v_camera_attfuse_2023_08_08_16_50_01/net_epoch_bestval_at17.pth'
att_state = torch.load(ATTFUSE_CHECKPOINT, map_location='cpu')
att_state = att_state.get('model_state_dict', att_state)
model_state = model.state_dict()
encoder_transfer = {k: v for k, v in att_state.items()
                    if k.startswith('encoder_m2.') and k in model_state
                    and tuple(v.shape) == tuple(model_state[k].shape)}
incompatible = model.load_state_dict(encoder_transfer, strict=False)
print('Transferred encoder tensors:', len(encoder_transfer))
print('Random/uninitialized-by-AttFuse modules include:',
      ['backbone_m2', 'pyramid_backbone', 'shrink_conv', 'detection heads'])
model = model.to(DEVICE)
optimizer = train_utils.setup_optimizer(hypes, model)
scheduler = train_utils.setup_lr_schedular(hypes, optimizer)
USE_AMP = True
GRADIENT_ACCUMULATION_STEPS = 2
GRADIENT_CLIP_NORM = 10.0
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

## Phase 3 — Inspect fusion and post-fusion instantiation

In [ ]:
encoder = model.encoder_m2
agent_backbone = model.backbone_m2
aligner = model.aligner_m2
pyramid_fusion = model.pyramid_backbone
post_fusion_shrinker = model.shrink_conv
post_fusion_heads = {'classification': model.cls_head,
                     'regression': model.reg_head,
                     'direction': model.dir_head}
print('Encoder:', type(encoder).__name__)
print('Agent backbone:', type(agent_backbone).__name__)
print('Aligner:', type(aligner).__name__)
print('Fusion:', type(pyramid_fusion).__name__)
print('Pyramid occupancy heads:', [type(getattr(pyramid_fusion, f'single_head_{i}')).__name__
                                    for i in range(pyramid_fusion.num_levels)])
print('Post-fusion:', type(post_fusion_shrinker).__name__,
      {k: type(v).__name__ for k, v in post_fusion_heads.items()})

## Phase 4 — Trace multiscale fusion and post-fusion outputs
At each pyramid level, a learned occupancy head scores every agent. Peer features and scores are affine-warped to ego; softmax-normalized occupancy scores weight the fused features. The multiscale decoder then joins the levels.

In [ ]:
def shape_of(value):
    if torch.is_tensor(value): return tuple(value.shape)
    if isinstance(value, dict): return {k: shape_of(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)): return [shape_of(v) for v in value]
    return type(value).__name__

trace = {}; handles = []
for name, module in [('LSS encoder', encoder), ('agent backbone', agent_backbone),
                     ('aligner', aligner),
                     ('post-fusion shrinker', post_fusion_shrinker),
                     *[(f'post-fusion {name}', module) for name, module in post_fusion_heads.items()]]:
    handles.append(module.register_forward_hook(
        lambda module, inputs, output, name=name: trace.update({name: shape_of(output)})))
for level in range(pyramid_fusion.num_levels):
    head = getattr(pyramid_fusion, f'single_head_{level}')
    handles.append(head.register_forward_hook(
        lambda module, inputs, output, level=level: trace.update({f'pyramid occupancy level {level}': shape_of(output)})))
handles.append(post_fusion_shrinker.register_forward_pre_hook(
    lambda module, inputs: trace.update({'PyramidFusion decoded output': shape_of(inputs[0])})))
batch = next(iter(train_loader)); batch = train_utils.to_device(batch, DEVICE)
model.eval()
with torch.inference_mode(), torch.cuda.amp.autocast(enabled=USE_AMP):
    output = model(batch['ego'])
for handle in handles: handle.remove()
for stage, shape in trace.items(): print(stage, '->', shape)
print('Pyramid level occupancy maps:', [tuple(x.shape) for x in output['occ_single_list']])
print('Post-fusion predictions:', {k: tuple(v.shape) for k, v in output.items() if torch.is_tensor(v)})
del output, batch, trace
gc.collect(); torch.cuda.empty_cache()

## Phase 5 — Detection loss + Pyramid occupancy loss + backward
The first term supervises fused detection heads. The `_single` term supervises the per-agent multiscale occupancy maps used as fusion weights.

In [ ]:
APPLY_OPTIMIZER_STEP = False
torch.cuda.empty_cache()
batch = train_utils.to_device(next(iter(train_loader)), DEVICE)
model.train(); optimizer.zero_grad(set_to_none=True); model.zero_grad(set_to_none=True)
with torch.cuda.amp.autocast(enabled=USE_AMP):
    output = model(batch['ego'])
    detection_loss = criterion(output, batch['ego']['label_dict'])
    occupancy_loss = criterion(output, batch['ego']['label_dict_single'], suffix='_single')
    total_loss = detection_loss + hypes['train_params'].get('single_weight', 1.0) * occupancy_loss
scaler.scale(total_loss).backward()
if APPLY_OPTIMIZER_STEP:
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
    scaler.step(optimizer); scaler.update()
print({'detection_loss': float(detection_loss.detach()),
       'pyramid_occupancy_loss': float(occupancy_loss.detach()),
       'total_loss': float(total_loss.detach()),
       'weights_updated': APPLY_OPTIMIZER_STEP})
optimizer.zero_grad(set_to_none=True); model.zero_grad(set_to_none=True)
del output, detection_loss, occupancy_loss, total_loss, batch
gc.collect(); torch.cuda.empty_cache()

## Phase 6 — Correct validation and controlled training loop

In [ ]:
SINGLE_WEIGHT = hypes['train_params'].get('single_weight', 1.0)
def pyramid_loss(active_model, active_criterion, batch):
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        result = active_model(batch['ego'])
        detection = active_criterion(result, batch['ego']['label_dict'])
        occupancy = active_criterion(result, batch['ego']['label_dict_single'], suffix='_single')
    return detection + SINGLE_WEIGHT * occupancy

def validate(model, loader):
    model.eval(); losses = []
    with torch.inference_mode():
        for item in loader:
            item = train_utils.to_device(item, DEVICE)
            losses.append(float(pyramid_loss(model, criterion, item)))
    return statistics.mean(losses)

print('Initial experimental validation loss:', validate(model, val_loader))

In [ ]:
RUN_FULL_TRAINING = False
RESUME_STATE = None
RUN_NAME = 'qcar_pyramid_' + datetime.now().strftime('%Y%m%d_%H%M%S')
SAVE_DIR = (Path(RESUME_STATE).parent if RESUME_STATE is not None
            else HEAL_ROOT/'opencood/logs'/RUN_NAME)
if RUN_FULL_TRAINING:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    with open(SAVE_DIR/'resolved_hypes.json', 'w') as stream:
        json.dump(hypes, stream, indent=2, default=str)
    best, start_epoch, history = float('inf'), 0, []
    if RESUME_STATE is not None:
        saved = torch.load(RESUME_STATE, map_location='cpu')
        model.load_state_dict(saved['model_state_dict'], strict=True)
        optimizer.load_state_dict(saved['optimizer_state_dict'])
        scheduler.load_state_dict(saved['scheduler_state_dict'])
        scaler.load_state_dict(saved['scaler_state_dict'])
        best = saved['best_val_loss']; start_epoch = saved['epoch'] + 1
        history = saved.get('history', [])
    for epoch in range(start_epoch, hypes['train_params']['epoches']):
        model.train(); epoch_losses = []
        optimizer.zero_grad(set_to_none=True); model.zero_grad(set_to_none=True)
        for step, item in enumerate(train_loader):
            item = train_utils.to_device(item, DEVICE)
            full_loss = pyramid_loss(model, criterion, item)
            scaled_loss = full_loss / GRADIENT_ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
            update_now = ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
                          or step + 1 == len(train_loader))
            if update_now:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True); model.zero_grad(set_to_none=True)
            epoch_losses.append(float(full_loss.detach()))
        val_loss = validate(model, val_loader)
        train_loss = statistics.mean(epoch_losses)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
        print(f'epoch {epoch}: train={train_loss:.6f} val={val_loss:.6f}')
        if val_loss < best:
            best = val_loss
            for old_best in SAVE_DIR.glob('net_epoch_bestval_at*.pth'): old_best.unlink()
            torch.save(model.state_dict(), SAVE_DIR/f'net_epoch_bestval_at{epoch+1}.pth')
        scheduler.step(epoch)
        training_state = {'epoch': epoch, 'model_state_dict': model.state_dict(),
                          'optimizer_state_dict': optimizer.state_dict(),
                          'scheduler_state_dict': scheduler.state_dict(),
                          'scaler_state_dict': scaler.state_dict(),
                          'best_val_loss': best, 'history': history}
        torch.save(training_state, SAVE_DIR/'training_state_last.pth')
        with open(SAVE_DIR/'training_history.json', 'w') as stream:
            json.dump(history, stream, indent=2)
        gc.collect(); torch.cuda.empty_cache()
    print('Pyramid training complete:', SAVE_DIR)
else:
    print('Production-style Pyramid training is ready but disabled.')

## Phase 7 — Final fit on all 113 development frames
Freeze the epoch count using development validation, then restart from the same AttFuse encoder initialization and fit once on all frames. No checkpoint selection occurs here.

In [ ]:
RUN_FINAL_FIT = False
SELECTED_FINAL_EPOCHS = None
if RUN_FINAL_FIT:
    assert isinstance(SELECTED_FINAL_EPOCHS, int) and SELECTED_FINAL_EPOCHS > 0
    final_hypes = copy.deepcopy(hypes)
    final_hypes['root_dir'] = hypes['_qcar_final_train_dir']
    final_dataset = build_dataset(final_hypes, visualize=False, train=True)
    final_loader = DataLoader(final_dataset, batch_size=1, shuffle=True, num_workers=0,
                              collate_fn=final_dataset.collate_batch_train,
                              pin_memory=True, drop_last=False)
    final_model = train_utils.create_model(final_hypes)
    final_model_state = final_model.state_dict()
    final_encoder_transfer = {k: v for k, v in att_state.items()
                              if k.startswith('encoder_m2.') and k in final_model_state
                              and tuple(v.shape) == tuple(final_model_state[k].shape)}
    final_model.load_state_dict(final_encoder_transfer, strict=False)
    final_model = final_model.to(DEVICE)
    final_criterion = train_utils.create_loss(final_hypes)
    final_optimizer = train_utils.setup_optimizer(final_hypes, final_model)
    final_scheduler = train_utils.setup_lr_schedular(final_hypes, final_optimizer)
    final_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    final_dir = HEAL_ROOT/'opencood/logs'/('qcar_pyramid_final_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    final_dir.mkdir(parents=True, exist_ok=False)
    for epoch in range(SELECTED_FINAL_EPOCHS):
        final_model.train(); final_optimizer.zero_grad(set_to_none=True)
        for step, item in enumerate(final_loader):
            item = train_utils.to_device(item, DEVICE)
            final_loss = pyramid_loss(final_model, final_criterion, item)
            accumulation_loss = final_loss / GRADIENT_ACCUMULATION_STEPS
            final_scaler.scale(accumulation_loss).backward()
            update_now = ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
                          or step + 1 == len(final_loader))
            if update_now:
                final_scaler.unscale_(final_optimizer)
                torch.nn.utils.clip_grad_norm_(final_model.parameters(), GRADIENT_CLIP_NORM)
                final_scaler.step(final_optimizer); final_scaler.update()
                final_optimizer.zero_grad(set_to_none=True)
        final_scheduler.step(epoch)
        print('final-fit epoch', epoch, 'loss', float(final_loss.detach()))
    final_checkpoint = final_dir/f'net_epoch{SELECTED_FINAL_EPOCHS}.pth'
    torch.save(final_model.state_dict(), final_checkpoint)
    with open(final_dir/'final_fit_protocol.json', 'w') as stream:
        json.dump({'frames': len(final_dataset), 'epochs': SELECTED_FINAL_EPOCHS,
                   'seed': SEED, 'encoder_transfer_tensors': len(final_encoder_transfer),
                   'checkpoint': str(final_checkpoint),
                   'test_policy': 'new independent trajectory'}, stream, indent=2)
    print('Final Pyramid checkpoint:', final_checkpoint)
else:
    print('Final fit disabled until SELECTED_FINAL_EPOCHS is frozen.')

## Interpretation boundary
A successful loss curve only proves that the experimental Pyramid model optimizes. It has no comparable pretrained fusion checkpoint and this trajectory cannot provide a final test. Compare against AttFuse on development validation, freeze decisions, then collect a new trajectory.